# BirdCLEF+ 2026 — Phase 3: Submission Notebook (Kaggle CPU, offline)

**目標**: 用凍結的 Perch v2 CPU 編碼器 + 本地訓好的 MLP head (Variant B, val macro ROC-AUC = 0.9646),對 `test_soundscapes/*.ogg` 產生 `submission.csv`。

## Kaggle Notebook 設定

- **Accelerator:** None (CPU only)
- **Internet:** OFF (提交時必須關掉)
- **Persistence:** Files only
- **Inputs** (四個):
  1. Competition: **BirdCLEF+ 2026** → `/kaggle/input/birdclef-2026/`
  2. Dataset: `perch-hoplite-wheels` → `/kaggle/input/perch-hoplite-wheels/` (離線 wheel 檔)
  3. Model: `perch-v2-cpu` → `/kaggle/input/perch-v2-cpu/` (Perch SavedModel)
  4. Dataset: `birdclef-mlp-head` → `/kaggle/input/birdclef-mlp-head/mlp_best.pt`

若你建立 Dataset/Model 的名稱不同,調整 Cell 2 的 `PATH_*` 常數。

## 與 Phase 1 一致的關鍵細節
- 同樣用 `perch_hoplite.zoo.model_configs.load_model_by_name` 載入模型,但名字用 `perch_v2_cpu` (權重相同、XLA 編譯平台不同)
- 同樣 5-sec 無重疊 window,`FRAME_LEN = sample_rate * 5 = 160000`
- 同樣 buffer 到 BATCH_SIZE=32 固定 shape 再呼叫 `model.batch_embed` (避 XLA 重複編譯)
- 類別欄位順序嚴格使用 `taxonomy.csv` 的 `primary_label` 欄 (和 Phase 1 的 `species_list` 建立方式一致)


## Cell 1 — 離線安裝 perch-hoplite + TF 2.20

Kaggle 預裝 TF 版本若低於 2.20 會讀不了 Perch v2 的 StableHLO ops。Wheels 需包さ `tensorflow==2.20.0` 與 `perch-hoplite` 及其依賴。

**跨了此 cell 必須 `Kernel → Restart & Clear Outputs` 再從 Cell 2 繼續。**

In [ ]:
!pip install -q --no-index --find-links /kaggle/input/perch-hoplite-wheels perch-hoplite


## Cell 2 — 環境檢查 + 路徑

重啟 kernel 後從這裡開始。完成後 assert 都要跳。

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU only

import time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
import torch
import torch.nn as nn
from tqdm.auto import tqdm

print("TF   :", tf.__version__)
print("Torch:", torch.__version__)

# —— Kaggle 路徑 (若 Dataset / Model 名稱不同請改這裡) ——
DATA_ROOT  = Path("/kaggle/input/birdclef-2026")
PERCH_DIR  = Path("/kaggle/input/perch-v2-cpu")      # Kaggle Model 挂載點
HEAD_DIR   = Path("/kaggle/input/birdclef-mlp-head")  # 5-fold weights 所在目錄
OUT_CSV    = Path("/kaggle/working/submission.csv")

N_FOLDS = 5

assert tf.__version__.startswith("2.20"), f"需 TF 2.20,目前 {tf.__version__} — 請重啟 kernel"
assert DATA_ROOT.exists(), "找不到競賽資料 — 在 Kaggle 右侧 Add Data 挂上 BirdCLEF+ 2026"
assert PERCH_DIR.exists(), f"找不到 Perch 模型 — 検查 Kaggle Model 挂載路徑 {PERCH_DIR}"
assert HEAD_DIR.exists(), f"找不到 MLP weights — 検查 Dataset 挂載路徑 {HEAD_DIR}"

# 檢查 5-fold weights 是否都在
for i in range(N_FOLDS):
    p = HEAD_DIR / f"mlp_fold{i}.pt"
    assert p.exists(), f"找不到 {p}"
print(f"found {N_FOLDS} fold weights in {HEAD_DIR}")

TEST_DIR = DATA_ROOT / "test_soundscapes"
test_files = sorted([p for p in TEST_DIR.glob("*.ogg")])
print(f"test files: {len(test_files)}")

## Cell 3 — 常數

In [ ]:
SR         = 32000               # Perch v2 採樣率
FRAME_SEC  = 5
FRAME_LEN  = SR * FRAME_SEC       # 160000
BATCH      = 32                   # 與 Phase 1 相同,避 XLA 重編譯
N_CLASSES  = 234
EMB_DIM    = 1536
TEST_LEN_SEC  = 60                # 每筆 test_soundscape 都是 1 分鐘
SEGS_PER_FILE = TEST_LEN_SEC // FRAME_SEC  # = 12


## Cell 4 — 類別欄位順序 (必須和訓練時一致)

Phase 1 用 `taxonomy_df["primary_label"].tolist()` 建立順序 — 這裡必須用完全相同的順序,否則 head 的 logit[i] 會對錯種。

In [ ]:
taxonomy_df = pd.read_csv(DATA_ROOT / "taxonomy.csv")
species_cols = taxonomy_df["primary_label"].tolist()
assert len(species_cols) == N_CLASSES, f"taxonomy 行數={len(species_cols)},期望={N_CLASSES}"

# 與 sample_submission 欄位相符 — 防御性檢查
sample = pd.read_csv(DATA_ROOT / "sample_submission.csv", nrows=1)
sample_species = [c for c in sample.columns if c != "row_id"]
assert set(sample_species) == set(species_cols), "species_cols 和 sample_submission 不一致"
# 最終提交欄位順序用 sample_submission 的順序，內部預測順序用 species_cols，寫檔時 reindex 對齊
print(f"species classes: {len(species_cols)}")


## Cell 5 — MLPHead 定義 + 載入 5-fold weights

架構與 `train_mlp_local.py` lines 125-140 相同。載入 `mlp_fold{0-4}.pt` (5-fold ensemble，各 fold 用 GroupKFold 訓練，帶 MixUp + 聲景加權)。推論時 5 個模型的 sigmoid 輸出取平均。

In [ ]:
class MLPHead(nn.Module):
    def __init__(self, in_dim=EMB_DIM, n_classes=N_CLASSES, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 768),
            nn.BatchNorm1d(768),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(768, 384),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(384, n_classes),
        )
    def forward(self, x):
        return self.net(x)

heads = []
for i in range(N_FOLDS):
    h = MLPHead()
    state = torch.load(HEAD_DIR / f"mlp_fold{i}.pt", map_location="cpu")
    h.load_state_dict(state)
    h.eval()
    heads.append(h)
    
total_params = sum(p.numel() for p in heads[0].parameters())
print(f"loaded {N_FOLDS} fold heads, {total_params:,} params each")

## Cell 6 — 載入 perch_v2_cpu + warm-up

`load_model_by_name` 會走 kagglehub 快取程式—因為此 notebook 無網路,所以我們把 Perch model 用 **Kaggle Model** 形式挂在 `/kaggle/input/perch-v2-cpu/`;設 `KAGGLEHUB_CACHE` 让 `kagglehub` 從本地檔案找。

> 如果你在 Kaggle 上傳模型時是放成 Dataset (而不是 Model) 或路徑結構不同,下面 `KAGGLEHUB_CACHE` 指向的路徑需要包含 `models/google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu/` 這種子目錄結構。

In [ ]:
# 让 kagglehub 從本地挂載點讀取
os.environ["KAGGLEHUB_CACHE"] = str(PERCH_DIR)

from perch_hoplite.zoo import model_configs

MODEL_NAME = "perch_v2_cpu"
t0 = time.time()
model = model_configs.load_model_by_name(MODEL_NAME)
print(f"Model loaded in {time.time()-t0:.1f}s, sample_rate={model.sample_rate}")
assert model.sample_rate == SR
assert model.batchable

# Warm-up — 觸發首次 XLA 編譯 (fixed shape = 32 x 160000)
t0 = time.time()
_ = model.batch_embed(np.zeros((BATCH, FRAME_LEN), dtype="float32"))
print(f"Warm-up (batch={BATCH}) in {time.time()-t0:.1f}s")


## Cell 7 — Framing helper

與 Phase 1 Cell 5 相同邏輯:
- 讀 ogg → float32 mono
- 長度 < 160000 → zero-pad
- 長度 ≥ 160000 → `len // FRAME_LEN` 個無重疊 window,末尾不足 5 秒舒棄 (test 音檔都是 60s 整數,続出 12 個 frame)

In [ ]:
def load_audio(fp):
    audio, sr = sf.read(str(fp), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    assert sr == SR, f"expected SR={SR} got {sr} for {fp}"
    return audio

def frame_audio(audio):
    if len(audio) < FRAME_LEN:
        return [np.pad(audio, (0, FRAME_LEN - len(audio)))]
    n = len(audio) // FRAME_LEN
    return [audio[i * FRAME_LEN:(i + 1) * FRAME_LEN] for i in range(n)]


## Cell 8 — 主迴圈: 抽 embedding → head 預測 → sigmoid

用一個總 buffer (跨檔) 來保證 batch 大小固定 32;最後不足 32 筆時 pad 到 32 再呼叫,取前 n 筆有效的。

In [ ]:
# 所有 (file_stem, end_sec) 的陠項產生器
rows = []                # [(file_stem, end_sec), ...]
all_probs = []           # list of (n, N_CLASSES) numpy arrays

buf_audio = []           # list of (160000,) float32
buf_rows  = []           # list of (stem, end_sec)

def flush(final=False):
    """把 buffer 里的 audio 撨進 model 抽 embedding + 5-fold head 預測,追加到 all_probs。"""
    if not buf_audio:
        return
    n_valid = len(buf_audio)
    # 不足 BATCH 就 pad 到 BATCH
    if n_valid < BATCH:
        pad = [np.zeros(FRAME_LEN, dtype="float32")] * (BATCH - n_valid)
        batch = np.stack(buf_audio + pad, axis=0).astype("float32")
    else:
        batch = np.stack(buf_audio, axis=0).astype("float32")
    out = model.batch_embed(batch)
    embs = out.embeddings[:n_valid, 0, 0, :].astype("float32")  # (n_valid, 1536)
    with torch.no_grad():
        embs_t = torch.from_numpy(embs)
        # 5-fold ensemble: average sigmoid outputs
        probs = sum(torch.sigmoid(h(embs_t)) for h in heads) / N_FOLDS
        probs = probs.numpy()
    all_probs.append(probs)
    rows.extend(buf_rows)
    buf_audio.clear()
    buf_rows.clear()

t0 = time.time()
for fp in tqdm(test_files, desc="test_soundscapes"):
    stem = fp.stem
    try:
        audio = load_audio(fp)
    except Exception as e:
        print(f"[skip] {stem}: {e}")
        for k in range(SEGS_PER_FILE):
            rows.append((stem, (k + 1) * FRAME_SEC))
            all_probs.append(np.zeros((1, N_CLASSES), dtype="float32"))
        continue
    frames = frame_audio(audio)
    for k, f in enumerate(frames[:SEGS_PER_FILE]):
        end_sec = (k + 1) * FRAME_SEC
        buf_audio.append(f)
        buf_rows.append((stem, end_sec))
        if len(buf_audio) >= BATCH:
            flush()
    if len(frames) < SEGS_PER_FILE:
        for k in range(len(frames), SEGS_PER_FILE):
            end_sec = (k + 1) * FRAME_SEC
            buf_audio.append(np.zeros(FRAME_LEN, dtype="float32"))
            buf_rows.append((stem, end_sec))
            if len(buf_audio) >= BATCH:
                flush()
flush(final=True)

probs_arr = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, N_CLASSES))

# Temporal smoothing: average each segment with its neighbors (same file)
if len(probs_arr) > 0:
    smoothed = probs_arr.copy()
    for i in range(0, len(smoothed), SEGS_PER_FILE):
        for j in range(SEGS_PER_FILE):
            k = i + j
            neighbors = [probs_arr[k]]
            if j > 0:
                neighbors.append(probs_arr[k - 1])
            if j < SEGS_PER_FILE - 1:
                neighbors.append(probs_arr[k + 1])
            smoothed[k] = np.mean(neighbors, axis=0)
    probs_arr = smoothed

print(f"inference done in {time.time()-t0:.1f}s, probs={probs_arr.shape}, rows={len(rows)}")

## Cell 9 — 寫 submission.csv

In [ ]:
assert len(rows) == probs_arr.shape[0] == len(test_files) * SEGS_PER_FILE, (
    f"row count mismatch: rows={len(rows)}, probs={probs_arr.shape[0]}, expected={len(test_files) * SEGS_PER_FILE}"
)

row_ids = [f"{stem}_{end_sec}" for stem, end_sec in rows]
df = pd.DataFrame(probs_arr, columns=species_cols)
df.insert(0, "row_id", row_ids)

# 重新對齊到 sample_submission 的欄位順序 (使用 sample_species 的順序)
df = df[["row_id"] + sample_species]

df.to_csv(OUT_CSV, index=False, float_format="%.6f")
print(f"wrote {OUT_CSV} shape={df.shape}")


## Cell 10 — Sanity checks

In [ ]:
check = pd.read_csv(OUT_CSV)
print("shape        :", check.shape)
print("columns head :", list(check.columns[:5]))
print("n species col:", check.shape[1] - 1)

prob_cols = [c for c in check.columns if c != "row_id"]
assert check.shape[1] == 1 + N_CLASSES, f"{check.shape[1]} != {1 + N_CLASSES}"
assert check.shape[0] == len(test_files) * SEGS_PER_FILE, (
    f"rows={check.shape[0]} expected={len(test_files) * SEGS_PER_FILE}"
)

if len(check) > 0:
    print("row_id sample:", check["row_id"].head(3).tolist())
    vals = check[prob_cols].to_numpy()
    print(f"prob min/max : {vals.min():.6f} / {vals.max():.6f}")
    print(f"prob mean    : {vals.mean():.6f}")
    assert vals.min() >= 0.0 and vals.max() <= 1.0
else:
    # Dry-run on Kaggle: test_soundscapes/ only has readme.txt. Empty CSV
    # (with correct header) is the expected and accepted output — at scoring
    # time Kaggle re-runs the notebook with the hidden test files in place.
    print("\n(empty CSV — expected for Kaggle dry-run before Submit)")

print("\n✓ submission.csv OK")
